# Vector Stores, Embeddings & RAG (Retrieval-Augmented Generation)

In [1]:
from sentence_transformers import SentenceTransformer

### Embeddings

Embeddings are numerical representations of data. Text, images and audio are stored as vectors.

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [
    "I think Machine Learning is great.",
    "Climate change is a global issue.",
    "Natural Language Processing is an important concept."
]

embeddings = model.encode(texts)
print(embeddings)

/Users/sanasalim/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[[-0.03421252 -0.08071415  0.03668646 ...  0.10850035 -0.00574369
   0.0077207 ]
 [ 0.03692482  0.07813372  0.1351478  ... -0.10862106 -0.02938253
   0.0205421 ]
 [ 0.04330918 -0.01448374  0.00250422 ...  0.12607285  0.02292273
  -0.01265183]]


In [3]:
print(embeddings.shape) # 384 dimensions per sentence

(3, 384)


### ChromaDB

An open-source vector database used to efficiently store, search and manage vector embeddings. It allows for fast similarity search.

In [4]:
import chromadb

In [5]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="personal_collection")
collection.add(
    documents=[
        "I think Machine Learning is great.",
        "Climate change is a global issue.",
        "Natural Language Processing is an important concept."
    ],
    ids=[
        "id1",
        "id2",
        "id3"
    ]
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [6]:
print(collection.count())

3


In [7]:
results = collection.query(
    query_texts=["Tell me about AI"],
    n_results=2
)

print(results["documents"])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[['I think Machine Learning is great.', 'Natural Language Processing is an important concept.']]


In [12]:
texts = ["I enjoy learning about Artificial Intelligence.",
        "Natural Language Processing is great.",
        "AI is pretty cool."]
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts).tolist()
collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=["id4","id5","id6"]
)

/Users/sanasalim/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [13]:
chroma_client.list_collections()

[Collection(name=personal_collection)]

In [21]:
query = "Tell me about AI"

results = collection.query(
    query_texts=[query],
    n_results=4
)

print(results["documents"])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[['AI is pretty cool.', 'I enjoy learning about Artificial Intelligence.', 'I think Machine Learning is great.', 'Natural Language Processing is an important concept.']]


In [49]:
from langchain_community.document_loaders import TextLoader

ai_loader = TextLoader(
    "/Users/sanasalim/Downloads/Linkific/Day 16/documents/ai.txt",encoding="utf-8"
)
ai_docs = ai_loader.load()
print(ai_docs)

[Document(metadata={'source': '/Users/sanasalim/Downloads/Linkific/Day 16/documents/ai.txt'}, page_content='Artificial intelligence (AI) is technology that enables computers and machines to simulate human learning, comprehension, problem solving, decision making, creativity and autonomy.\nDirectly underneath AI, we have machine learning, which involves creating models by training an algorithm to make predictions or decisions based on data. It encompasses a broad range of techniques that enable computers to learn from and make inferences based on data without being explicitly programmed for specific tasks.\nDeep learning is a subset of machine learning that uses multilayered neural networks, called deep neural networks, that more closely simulate the complex decision-making power of the human brain.')]


In [50]:
leave_loader = TextLoader(
    "/Users/sanasalim/Downloads/Linkific/Day 16/documents/linkific_leave.txt",encoding="utf-8"
)
leave_docs = leave_loader.load()
print(leave_docs)

[Document(metadata={'source': '/Users/sanasalim/Downloads/Linkific/Day 16/documents/linkific_leave.txt'}, page_content='Linkific company policy allows for 12 days of leave.')]


In [52]:
cc_loader = TextLoader(
    "/Users/sanasalim/Downloads/Linkific/Day 16/documents/climate change.txt",encoding="utf-8"
)
cc_docs = cc_loader.load()
print(cc_docs)
docs = ai_docs + leave_docs + cc_docs

[Document(metadata={'source': '/Users/sanasalim/Downloads/Linkific/Day 16/documents/climate change.txt'}, page_content='Climate change refers to long-term shifts in temperatures and weather patterns. Such shifts can be natural, due to changes in the sun’s activity or large volcanic eruptions. But since the 1800s, human activities have been the main driver of climate change, primarily due to the burning of fossil fuels like coal, oil and gas.\nBurning fossil fuels generates greenhouse gas emissions that act like a blanket wrapped around the Earth, trapping the sun’s heat and raising temperatures.\nThe main greenhouse gases that are causing climate change include carbon dioxide and methane. These come from using gasoline for driving a car or coal for heating a building, for example. Clearing land and cutting down forests can also release carbon dioxide. Agriculture, oil and gas operations are major sources of methane emissions. Energy, industry, transport, buildings, agriculture and land

"Chunking” refers to the segmentation of input text into shorter and more meaningful units. This allows the system to efficiently pinpoint and retrieve relevant pieces of information

In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=100)
chunks = splitter.split_documents(docs)

In [54]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Users/sanasalim/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [55]:
from langchain_community.vectorstores import Chroma
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [64]:
from langchain_classic import hub
#prompt = hub.pull("rlm/rag-prompt")
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
"""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:""")
import os
from dotenv import load_dotenv
load_dotenv() 
api_key = os.environ.get("GROQ_API_KEY")
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile",api_key=api_key,temperature=0.8)

In [67]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
retriever = vector_store.as_retriever()
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [68]:
question = "How many leave days do linkific employees get?"

docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in docs
)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
